# Setup Environment and Imports
In this section, we import all necessary libraries and create the required directories for saving our ensemble LoRA models.

In [1]:
import os
import html
import re
import pandas as pd
import numpy as np
import torch
import random
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from peft import get_peft_model, LoraConfig, TaskType
from peft import PeftModel, PeftConfig
from sklearn.metrics import classification_report, f1_score

# Ensure output directory exists
os.makedirs("ensemble_loras", exist_ok=True)


/vol/bitbucket/hl3025/nlp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load and Prepare Data
Here we load the training and validation datasets directly from `dontpatronizeme_pcl.tsv` using custom parsing logic, avoiding external library dependency. Finally, we construct our train and dev pandas DataFrames.

In [2]:
# 1. LOAD DATA
def clean_text(text: str) -> str:
    text = html.unescape(text)                       # &amp; → & etc.
    text = re.sub(r"<[^>]+>", " ", text)             # strip HTML tags
    text = re.sub(r"[\n\r]", " ", text)              # remove embedded newlines
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_task1(train_path):
    rows=[]
    with open(os.path.join(train_path, 'dontpatronizeme_pcl.tsv')) as f:
        # Skip top 4 rows as per custom original loader
        for line in f.readlines()[4:]:
            parts = line.split('\t')
            lbin = 0 if parts[-1].strip() in ['0', '1'] else 1
            text = parts[4] if len(parts) > 4 else ""
            rows.append({
                'par_id': parts[0].strip(),
                'keyword': parts[2].strip() if len(parts) > 2 else "",
                'text': clean_text(text), 
                'label': lbin, 
            })
    return pd.DataFrame(rows)

data = load_task1('.')

trids = pd.read_csv('train_semeval_parids-labels.csv')
teids = pd.read_csv('dev_semeval_parids-labels.csv')
trids.par_id = trids.par_id.astype(str)
teids.par_id = teids.par_id.astype(str)

# Build DataFrame Task 1 Function
def build_df(ids_df):
    rows = []
    for idx in range(len(ids_df)):
        parid = ids_df.par_id.iloc[idx]
        row_data = data.loc[data.par_id == parid]
        if not row_data.empty:
            rows.append({
                'par_id': parid,
                'community': row_data.keyword.values[0],
                'text': row_data.text.values[0],
                'label': int(row_data.label.values[0])
            })
    return pd.DataFrame(rows)

trdf1 = build_df(trids)
tedf1 = build_df(teids)
tedf1 = tedf1.sample(frac=1, random_state=42).reset_index(drop=True)

print("Training set size:", len(trdf1))
print("Dev set size:", len(tedf1))


Training set size: 8375
Dev set size: 2094


# Load Test Data
We also load the actual test dataset (`task4_test.tsv`) which needs predictions.

In [3]:
# Load the test set
# The test set is a TSV without a header.
with open('task4_test.tsv', 'r', encoding='utf-8') as f:
    test_lines = f.readlines()

test_rows = []
for line in test_lines:
    parts = line.split('\t')
    if len(parts) >= 5:
        # Assuming the text is the 5th column
        test_rows.append({'par_id': parts[0].strip(), 'text': clean_text(parts[4])})
    elif len(parts) > 0:
        test_rows.append({'par_id': parts[0].strip(), 'text': ""})

test_df = pd.DataFrame(test_rows)
print("Test set size:", len(test_df))


Test set size: 3832


# Dataset and Tokenizer Setup
We define our PyTorch `Dataset` wrapper class and initialize the RoBERTa tokenizer.

In [4]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df['text'].astype(str).tolist()
        if 'label' in df.columns:
            self.labels = torch.tensor(df['label'].tolist(), dtype=torch.long)
        else:
            self.labels = None
        
        self.encodings = tokenizer(
            self.texts, truncation=True, padding="max_length", 
            max_length=max_length, return_tensors="pt"
        )
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, i):
        item = {
            "input_ids": self.encodings["input_ids"][i],
            "attention_mask": self.encodings["attention_mask"][i],
        }
        if self.labels is not None:
             item["labels"] = self.labels[i]
        return item

task1_dev_dataset = PCLDataset(tedf1, tokenizer)
task1_test_dataset = PCLDataset(test_df, tokenizer)

def compute_metrics_task1(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1).flatten()
    return {"f1": f1_score(labels, preds, average="binary", zero_division=0)}

def tune_threshold(probs, true_labels):
    best_f1, best_theta = 0.0, 0.5
    for theta in np.arange(0.1, 0.9, 0.01):
        preds = (probs >= theta).astype(int)
        f1 = f1_score(true_labels, preds, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_theta = f1, theta
    return best_theta, best_f1


# Ensemble Training Loop
In this section, we train an ensemble of multiple LoRA adapters. Each model gets a different random subset of negative examples to balance out the highly imbalanced positive class. We save each independently trained LoRA adapter.

In [5]:
# 2. ENSEMBLE TRAINING LOOP (BOOTSTRAPPING & DOWNSAMPLING)
NUM_MODELS = 5
lora_paths = []

for i in range(NUM_MODELS):
    print(f"\n=============================================")
    print(f"   TRAINING LORA MODEL {i+1}/{NUM_MODELS} ")
    print(f"=============================================")
    
    # BOOTSTRAPPING / DOWNSAMPLING STRATEGY:
    # We take ALL positive examples.
    # We randomly sample negative examples (ratio 1:2) using a COMPLETY DIFFERENT RANDOM SEED each time!
    pcldf = trdf1[trdf1.label == 1]
    negdf = trdf1[trdf1.label == 0]
    
    # Different random seed per model ensures different negative examples!
    sampled_neg = negdf.sample(n=len(pcldf)*2, random_state=42+i*10)
    
    # Combine and shuffle
    training_set = pd.concat([pcldf, sampled_neg]).sample(frac=1, random_state=42+i*10).reset_index(drop=True)
    task1_train_dataset = PCLDataset(training_set, tokenizer)
    
    # Initialize fresh base model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
    )
    
    # Initialize fresh LoRA attached to it
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=16,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "value", "key", "dense"]
    )
    model = get_peft_model(model, peft_config)
    
    training_args = TrainingArguments(
        output_dir=f"./ensemble_loras/model_{i}",
        num_train_epochs=10,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=1e-4, 
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=task1_train_dataset,
        compute_metrics=compute_metrics_task1,
    )
    
    trainer.train()
    
    # Save the tiny LoRA adapter weights specifically
    lora_path = f"./ensemble_loras/lora_adapter_{i}"
    model.save_pretrained(lora_path)
    lora_paths.append(lora_path)
    print(f"Saved LoRA Adapter {i+1} to {lora_path}!")

# To save the WHOLE ensemble together we can save our list of paths
np.save("ensemble_loras/lora_paths.npy", lora_paths)
print("Finished Training all models! Paths saved.")



   TRAINING LORA MODEL 1/5 


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 495.88it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Step,Training Loss
500,0.462961
1000,0.337669
1500,0.248465
2000,0.172234
2500,0.131796


Saved LoRA Adapter 1 to ./ensemble_loras/lora_adapter_0!

   TRAINING LORA MODEL 2/5 


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1388.80it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

...

Step,Training Loss
500,0.480491
1000,0.341060
1500,0.259659
2000,0.188266
2500,0.139367


Saved LoRA Adapter 4 to ./ensemble_loras/lora_adapter_3!

   TRAINING LORA MODEL 5/5 


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1368.21it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Step,Training Loss
500,0.467367
1000,0.317270
1500,0.253618
2000,0.181575
2500,0.113525


Saved LoRA Adapter 5 to ./ensemble_loras/lora_adapter_4!
Finished Training all models! Paths saved.


# Ensemble Inference
Now we take our set of trained LoRA adapters, inject them into our `base_model` one at a time, and record their predicted probabilities for both the dev and the test datasets.

In [6]:
# 3. ENSEMBLE INFERENCE (SOFT VOTING)
print("\n=============================================")
print("   PERFORMING ENSEMBLE INFERENCE (SOFT VOTING) ")
print("=============================================\n")

# Load exactly ONE heavy base model for inference
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)

true_labels1 = task1_dev_dataset.labels.numpy()

dev_all_probs = []
test_all_probs = []

# Iterate through the saved adapters, attaching them to the single base model one by one
for i, lora_path in enumerate(lora_paths):
    print(f"Loading LoRA {i+1} and generating logits...")
    peft_model = PeftModel.from_pretrained(base_model, lora_path)
    
    # Setup a trainer purely for prediction extraction
    infer_trainer = Trainer(model=peft_model)
    
    # Dev Predictions
    dev_preds = infer_trainer.predict(task1_dev_dataset)
    dev_logits = torch.tensor(dev_preds.predictions)
    dev_probs = torch.softmax(dev_logits, dim=-1)[:, 1].numpy()
    dev_all_probs.append(dev_probs)
    
    # Test Predictions
    test_preds = infer_trainer.predict(task1_test_dataset)
    test_logits = torch.tensor(test_preds.predictions)
    test_probs = torch.softmax(test_logits, dim=-1)[:, 1].numpy()
    test_all_probs.append(test_probs)

# 4. SOFT VOTING 
# Mathematically average the probability arrays across all models!
dev_avg_probs = np.mean(dev_all_probs, axis=0)
test_avg_probs = np.mean(test_all_probs, axis=0)

# Optimization: Find threshold that maximizes F1 using the ensemble average probabilities on Dev Set
best_theta, best_f1 = tune_threshold(dev_avg_probs, true_labels1)
print(f"\nBest threshold on Ensembled Validation Set: {best_theta:.2f} (F1: {best_f1:.4f})")



   PERFORMING ENSEMBLE INFERENCE (SOFT VOTING) 



Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1337.67it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Loading LoRA 1 and generating logits...


Loading LoRA 2 and generating logits...


/vol/bitbucket/hl3025/nlp_env/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


...

Loading LoRA 4 and generating logits...


/vol/bitbucket/hl3025/nlp_env/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading LoRA 5 and generating logits...


/vol/bitbucket/hl3025/nlp_env/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



Best threshold on Ensembled Validation Set: 0.89 (F1: 0.5758)


# Save Dev and Test Predictions
Save `dev.txt` and `test.txt` with exactly one prediction per line (0 or 1).

In [7]:
# Final Predictions
dev_final_preds = (dev_avg_probs >= best_theta).astype(int)
test_final_preds = (test_avg_probs >= best_theta).astype(int)

# Write dev.txt
with open('dev.txt', 'w') as f:
    for pred in dev_final_preds:
        f.write(f"{pred}\n")
print(f"Saved {len(dev_final_preds)} predictions to dev.txt")

# Write test.txt
with open('test.txt', 'w') as f:
    for pred in test_final_preds:
        f.write(f"{pred}\n")
print(f"Saved {len(test_final_preds)} predictions to test.txt")


Saved 2094 predictions to dev.txt
Saved 3832 predictions to test.txt
